# Postprocessing pipeline: from Generation Output to Final Graphs

This notebook is a self-contained reproducibility guide for the analysis pipeline used in the guidance study. It starts from raw MatterGen output archives and produces the downstream objects needed for the paper-level analysis:

1. postprocessed datasets;
2. target-descriptor histograms;
3. Pareto fronts in guidance loss vs. convex-hull energy;
4. raw-stage guided-vs-non-guided enrichment and the corresponding one-sided two-proportion p-value.

The test data are distributed with `vsbtools` under:

```text
vsbtools/materials_dataset/Examples/raw_generations/Cu-Si-P
```

The example contains one non-guided Cu-Si-P generation and guided generations targeting different Cu-P coordination numbers.

## 0. Required packages and environments

This notebook should be run from a **vsbtools environment**. Paths to the external MatterScout/MatterGen and GRACE environments are configured in the first code cell, so no machine-specific shell environment is required.

The intended setup has three components:

1. `vsbtools`: installed from the GitHub repository containing this notebook/data pipeline. This is the active Jupyter kernel.
2. `scout-matter`: MatterGen fork from <https://github.com/link-lab3629/scout-matter>, used for descriptor and guidance-loss functions.
3. `tensorpotential`: used by the GRACE estimator through the `vsbtools` GRACE bridge.

Recommended separation:

```bash
# 1. Analysis/kernel environment
python -m venv .venvs/vsbtools
source .venvs/vsbtools/bin/activate
pip install -U pip
pip install -e /path/to/vsbtools[materials_dataset]
python -m ipykernel install --user --name vsbtools --display-name vsbtools

# 2. MatterScout/MatterGen environment or source tree
# Install scout-matter following https://github.com/link-lab3629/scout-matter
# Then set MATTER_SCOUT_PATH in the first code cell to either the source tree,
# a venv root, or that venv's site-packages directory.

# 3. GRACE/tensorpotential environment
python -m venv .venvs/grace
source .venvs/grace/bin/activate
pip install tensorpotential
# Then set GRACE_ENV_OR_PYTHON in the first code cell to either the venv root
# or its bin/python executable.
```

If the notebook variables are left as `None`, `vsbtools` falls back to `MATTERGEN_PYTHON_PATH`, `GRACE_PYTHON`, or persistent local configuration in `~/.config/vsbtools/external_paths.json`.

For the fully contained setup used by reviewers, run the script distributed beside this notebook:

```bash
bash setup_reproducibility_envs.sh --root ./vsbtools_reproducibility_env --run-root ./vsbtools_reproducibility_run
```

That script clones `vsbtools` and `scout-matter`, creates all three venvs under `--root`, writes Jupyter/IPython/vsbtools state under `--root/state`, copies this notebook to `--root/work`, writes reproducibility outputs under `--run-root`, and launches JupyterLab from the contained `vsbtools` venv.


In [ ]:
from __future__ import annotations

import json
import math
import os
import subprocess
import sys
import warnings
from contextlib import contextmanager
from pathlib import Path

QUIET_TENSORFLOW = True             # suppress TensorFlow/tensorpotential startup logs
QUIET_OPTIONAL_IMPORT_WARNINGS = True  # suppress optional torch-geometric/tqdm notebook warnings

if QUIET_TENSORFLOW:
    os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
    os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
    warnings.filterwarnings(
        "ignore",
        message=".*enabling the new type promotion.*",
    )

if QUIET_OPTIONAL_IMPORT_WARNINGS:
    warnings.filterwarnings(
        "ignore",
        message="An issue occurred while importing '.*'. Disabling its usage.*",
        category=UserWarning,
    )
    warnings.filterwarnings(
        "ignore",
        message="IProgress not found.*",
        category=Warning,
    )

import vsbtools

# Optional per-notebook external dependency paths. These override environment
# variables for this kernel only. Leave as None to use MATTERGEN_PYTHON_PATH,
# GRACE_PYTHON, or ~/.config/vsbtools/external_paths.json.
MATTER_SCOUT_PATH = None       # e.g. Path('/path/to/scout-matter') or Path('/path/to/matter-scout-venv')
GRACE_ENV_OR_PYTHON = None     # e.g. Path('/path/to/grace-venv') or Path('/path/to/grace-venv/bin/python')


def _path_or_none(value):
    if value in (None, ""):
        return None
    return Path(value).expanduser().resolve()


def _python_from_venv(path: Path) -> Path:
    python = path / "bin" / "python"
    if not python.is_file():
        raise FileNotFoundError(python)
    return python


def _site_packages_from_python(python: Path) -> Path:
    code = 'import sysconfig; print(sysconfig.get_paths()["purelib"])'
    proc = subprocess.run([python.as_posix(), "-c", code], text=True, capture_output=True)
    if proc.returncode != 0:
        detail = proc.stderr.strip().splitlines()[-1] if proc.stderr.strip() else f"exit code {proc.returncode}"
        raise RuntimeError(f"Could not query site-packages from {python}: {detail}")
    site_packages = Path(proc.stdout.strip()).resolve()
    if not site_packages.is_dir():
        raise FileNotFoundError(site_packages)
    return site_packages


def _matter_scout_import_path(path_value) -> Path | None:
    path = _path_or_none(path_value)
    if path is None:
        return None
    if not path.exists():
        raise FileNotFoundError(path)
    if path.is_file():
        return _site_packages_from_python(path)
    if (path / "pyvenv.cfg").is_file():
        return _site_packages_from_python(_python_from_venv(path))
    return path


def _python_from_venv_or_executable(path_value) -> Path | None:
    path = _path_or_none(path_value)
    if path is None:
        return None
    if path.is_dir():
        path = path / "bin" / "python"
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


configured_matter_scout_path = _matter_scout_import_path(MATTER_SCOUT_PATH)
configured_grace_python = _python_from_venv_or_executable(GRACE_ENV_OR_PYTHON)

if configured_matter_scout_path is not None:
    os.environ["MATTERGEN_PYTHON_PATH"] = configured_matter_scout_path.as_posix()
if configured_grace_python is not None:
    os.environ["GRACE_PYTHON"] = configured_grace_python.as_posix()

VSBTOOLS_PACKAGE = Path(vsbtools.__file__).resolve().parent
VSBTOOLS_REPO = VSBTOOLS_PACKAGE.parent

print("Python:", sys.executable)
print("vsbtools package:", VSBTOOLS_PACKAGE)
print("Configured MatterScout path:", configured_matter_scout_path or "<using env/config fallback>")
print("Configured GRACE Python:", configured_grace_python or "<using env/config fallback>")
print("Effective MATTERGEN_PYTHON_PATH:", os.environ.get("MATTERGEN_PYTHON_PATH", "<not set>"))
print("Effective GRACE_PYTHON:", os.environ.get("GRACE_PYTHON", "<not set>"))


## 1. Configuration

The default configuration uses the packaged Cu-Si-P example. For another system, replace `INPUT_DIRS`, `SYSTEM`, `BOND`, and `TARGET` with the corresponding raw MatterGen output directories and target descriptor.

Input conventions:

- non-guided generation: pass one MatterGen output folder/archive;
- guided multibatch generation: pass the parent folder/archive containing `run_1`, `run_2`, ... as produced by `mattergen/tests/multiple_runs.sh`.

By default, the notebook writes derived files into `reproducibility_run/` next to this notebook. The contained setup script sets `VSBTOOLS_REPRO_RUN_ROOT`, so its outputs go to `vsbtools_reproducibility_run/` next to the environment workspace. No manuscript-local directories are required.

In [ ]:
from importlib.resources import files

DATASET_PACKAGE = "vsbtools.materials_dataset"
EXAMPLES_ROOT = Path(str(files(DATASET_PACKAGE).joinpath("Examples")))
RAW_GENERATION_ROOT = EXAMPLES_ROOT / "raw_generations" / "Cu-Si-P"
SCENARIO_FILE = EXAMPLES_ROOT / "scenario_no_relax.yaml"

NOTEBOOK_DIR = Path.cwd().resolve()
RUN_ROOT = Path(os.environ.get("VSBTOOLS_REPRO_RUN_ROOT", NOTEBOOK_DIR / "reproducibility_run")).expanduser().resolve()
POSTPROCESS_ROOT = RUN_ROOT / "MG_postprocess_pipelines"
PROCESSED_ROOT = POSTPROCESS_ROOT / "PROCESSED"
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)

# Packaged test MatterGen outputs.
INPUT_DIRS = [
    RAW_GENERATION_ROOT / "Cu-Si-P_nonguided.zip",
    RAW_GENERATION_ROOT / "Cu-Si-P_guided_CuP3.zip",
]

# Descriptor / target used for histogram and p-value.
SYSTEM = "Cu-Si-P"
GUIDANCE_KIND = "mean_coordination"  # implemented here: "mean_coordination" or "volume_pa"
BOND = "Cu-P"                         # used only for coordination targets
TARGET = 3.0
TARGET_HALF_WIDTH = 0.2
COORD_ALPHA = 200.0                    # abrupt coordination-counting limit; r_cut is recovered from parse_raw metadata when present

# Stages and plotting options.
HIST_STAGE = "symmetrize_raw"
PVALUE_STAGE = "symmetrize_raw"
REFERENCE_STAGE_FOR_PLOTS = "poll_db"
PARETO_STAGES = ["add_ref_deduplicated"]
N_PARETO_FRONTS = 3
SUPPRESS_PIPELINE_STDERR = True        # hides repeated spglib/TensorFlow/tensorpotential stderr noise during postprocessing
TRIM_LOSS = None                       # e.g. 0.1; None keeps plot_pareto defaults
TRIM_EHULL = 0.3                       # eV/atom display cutoff for Pareto plots

OUTPUT_DIR = RUN_ROOT / "analysis_outputs" / SYSTEM
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [EXAMPLES_ROOT, RAW_GENERATION_ROOT, SCENARIO_FILE, *INPUT_DIRS]:
    print(path, "exists=", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

## 2. Postprocess raw MatterGen outputs

This is the same pipeline entry point used by the manuscript postprocessing scripts, but called directly from `vsbtools` so the notebook is portable.

The scenario YAML controls the postprocessing sequence. In the packaged example it includes the operations required to reach the final analysis stages, including GRACE-based energy estimation through the `GRACE_PYTHON` bridge.

In [ ]:
from tempfile import TemporaryDirectory
from zipfile import ZipFile

from vsbtools.materials_dataset.analysis.scenario_pipeline import process_generation_dir
from vsbtools.materials_dataset.io.structures_dataset_io import exploded_zip_tree

BATCH_METADATA_FILE = "input_parameters.txt"


@contextmanager
def suppress_process_stderr(enabled: bool):
    if not enabled:
        yield
        return

    sys.stderr.flush()
    saved_stderr = os.dup(2)
    try:
        with open(os.devnull, "w") as devnull:
            os.dup2(devnull.fileno(), 2)
            yield
    finally:
        os.dup2(saved_stderr, 2)
        os.close(saved_stderr)


def is_multibatch_dir(path: Path) -> bool:
    return (path / "run_1").is_dir()


def list_processed_repos() -> list[Path]:
    if not PROCESSED_ROOT.exists():
        return []
    return sorted(path for path in PROCESSED_ROOT.iterdir() if path.is_dir())


def run_pipeline_on_directory(generation_dir: Path) -> None:
    with suppress_process_stderr(SUPPRESS_PIPELINE_STDERR):
        repo = process_generation_dir(
            generation_dir,
            PROCESSED_ROOT,
            SCENARIO_FILE,
            batch_metadata_file=BATCH_METADATA_FILE,
        )
    if repo is None:
        raise RuntimeError(
            f"No {BATCH_METADATA_FILE!r} file was found under {generation_dir}. "
            "This directory is not recognized as a MatterGen output."
        )


def process_one_generation(input_path: Path) -> None:
    input_path = Path(input_path).expanduser().resolve()
    if not input_path.exists():
        raise FileNotFoundError(input_path)

    layout = "guided multibatch" if input_path.is_dir() and is_multibatch_dir(input_path) else "single generation/archive"
    print(f"Processing {layout}: {input_path}")

    if input_path.is_file() and input_path.suffix.lower() == ".zip":
        # A single MatterGen output archive is itself the generation source.
        # `exploded_zip_tree` is for directories containing archives; passing a
        # zip file to it would silently produce an empty temporary tree.
        with TemporaryDirectory(prefix=f"{input_path.stem}_") as tmp:
            extracted = Path(tmp) / input_path.stem
            extracted.mkdir(parents=True, exist_ok=True)
            with ZipFile(input_path) as zf:
                zf.extractall(extracted)
            run_pipeline_on_directory(extracted);
    elif input_path.is_dir():
        # For a directory tree that may contain nested zip archives, preserve
        # the existing utility behavior.
        with exploded_zip_tree(input_path) as tmp_inbox:
            if any(tmp_inbox.rglob(BATCH_METADATA_FILE)):
                run_pipeline_on_directory(tmp_inbox);
            else:
                run_pipeline_on_directory(input_path);
    else:
        raise ValueError(f"Unsupported input path: {input_path}")


for input_dir in INPUT_DIRS:
    process_one_generation(input_dir);

processed_repos = list_processed_repos();
print("Processed repositories created/found under", PROCESSED_ROOT)
for repo in processed_repos:
    print(" -", repo.name)

if not processed_repos:
    raise RuntimeError(
        "Postprocessing did not create any processed repository. "
        "Check the messages from this cell first, especially missing bridge configuration "
        "for MATTERGEN_PYTHON_PATH or GRACE_PYTHON. Section 3 cannot run until this directory is populated."
    )

## 3. Locate the processed system repository and build summary artifacts

`build_guidance_summary_for_processed_system` creates the auxiliary CSV/TXT/POSCAR artifacts used downstream for Pareto analysis and quality tables. The processed system directory is normalized by sorting element symbols, matching the postprocessing convention.

In [ ]:
from vsbtools.materials_dataset.scripts.build_tables import (
    build_guidance_summary_for_processed_system,
    collect_guidance_summary_artifacts,
    GuidanceSummaryBuildReport,
)


def normalized_system_name(system: str) -> str:
    return "-".join(sorted(system.split("-")))


def list_processed_repos() -> list[Path]:
    if not PROCESSED_ROOT.exists():
        return []
    return sorted(path for path in PROCESSED_ROOT.iterdir() if path.is_dir())


def processed_system_dir(system: str) -> Path:
    available = list_processed_repos()
    normalized = normalized_system_name(system)
    direct = PROCESSED_ROOT / normalized
    if direct.exists():
        return direct

    requested_elements = set(system.split("-"))
    element_matches = [
        repo for repo in available
        if set(repo.name.split("-")) == requested_elements
    ]
    if len(element_matches) == 1:
        return element_matches[0]

    available_names = [repo.name for repo in available]
    raise FileNotFoundError(
        f"Processed system directory for {system!r} was not found under {PROCESSED_ROOT}.\n"
        f"Expected normalized name: {normalized!r}.\n"
        f"Available processed repositories: {available_names or '<none>'}.\n"
        "If the list is empty, run Section 2 and resolve any postprocessing/bridge errors before continuing."
    )


SYSTEM_DIR = processed_system_dir(SYSTEM)

summary_report = build_guidance_summary_for_processed_system(
    SYSTEM_DIR,
    target_stages=PARETO_STAGES,
    auto_ref_stages=True,
    max_pareto_front=N_PARETO_FRONTS,
    return_report=True,
)

print(f"Built guidance summary artifacts for {len(summary_report.generation_reports)} generation repositories")
print(f"Pareto stages: {', '.join(PARETO_STAGES)}; max fronts: {N_PARETO_FRONTS}")

# To inspect already-created artifact paths without rebuilding, use:
# generation_reports = [
#     collect_guidance_summary_artifacts(repo, target_stages=PARETO_STAGES)
#     for repo in sorted(SYSTEM_DIR.iterdir())
#     if repo.is_dir()
# ]
# GuidanceSummaryBuildReport({}, processed_system_repo=SYSTEM_DIR, generation_reports=generation_reports)

## 4. Histogram of the guided descriptor

This uses the article plotting helpers from `vsbtools.materials_dataset.analysis.guidance_statistics`. For coordination targets, the helper searches for processed repositories whose parse-raw metadata matches the selected target and automatically includes the corresponding non-guided dataset.

In [ ]:
import matplotlib.pyplot as plt

from vsbtools.materials_dataset.analysis.guidance_statistics import (
    get_mean_coordination_gen_dirs,
    get_volume_pa_gen_dirs,
    plot_av_env_guidance_results,
    plot_volume_pa_guidance_results,
)


def get_generation_dirs_for_target():
    if GUIDANCE_KIND == "mean_coordination":
        return get_mean_coordination_gen_dirs(PROCESSED_ROOT, SYSTEM, bond=BOND, target=TARGET)
    if GUIDANCE_KIND == "volume_pa":
        return get_volume_pa_gen_dirs(PROCESSED_ROOT, SYSTEM, guidance_name="volume_pa", target=TARGET)
    raise ValueError(f"Unsupported GUIDANCE_KIND for this notebook: {GUIDANCE_KIND!r}")


GEN_DIRS = get_generation_dirs_for_target()
print("Matched generation repositories:")
for gen_dir in GEN_DIRS:
    print(" -", gen_dir.name)

if GUIDANCE_KIND == "mean_coordination":
    ax = plot_av_env_guidance_results(
        PROCESSED_ROOT,
        SYSTEM,
        BOND,
        target=TARGET,
        stage=HIST_STAGE,
        filter_max_el=False,
        show_gaussian=True,
        other_callable_params={"alpha": COORD_ALPHA},
        half_width=TARGET_HALF_WIDTH,
    )
elif GUIDANCE_KIND == "volume_pa":
    ax = plot_volume_pa_guidance_results(
        PROCESSED_ROOT,
        SYSTEM,
        target=TARGET,
        stage=HIST_STAGE,
        filter_max_el=False,
        show_gaussian=True,
        half_width=TARGET_HALF_WIDTH,
    )

hist_path = OUTPUT_DIR / f"{SYSTEM}_{GUIDANCE_KIND}_{HIST_STAGE}_histogram.pdf"
plt.savefig(hist_path, bbox_inches="tight", pad_inches=0.1)
print("Saved:", hist_path)

## 5. Raw-stage enrichment and p-value

The enrichment test uses the full raw/symmetrized stage as denominator:

$$
\frac{|C_G|}{|R_G|} \quad \text{vs.} \quad \frac{|C_{NG}|}{|R_{NG}|},
$$

where `C` is the set of structures satisfying the target constraint and `R` is the full generated set at `PVALUE_STAGE`. No exact-element-count filtering is applied to the denominators. The p-value is the one-sided pooled two-proportion z-test with alternative `guided > non-guided`.

In [ ]:
import pandas as pd

from vsbtools.materials_dataset.analysis.guidance_statistics import (
    callables_from_ds,
    collect_stage_dataset_dict,
    get_two_proportion_z_test,
)


def label_is_reference(label: str) -> bool:
    return label.lower() == "reference"


def label_is_nonguided(label: str) -> bool:
    return label == "Non-guided"


def first_guided_parse_raw_dataset(gen_dirs):
    ds_dict_parse = collect_stage_dataset_dict(gen_dirs, stage="parse_raw", ref_stage=REFERENCE_STAGE_FOR_PLOTS)
    for label, ds in ds_dict_parse.items():
        if not label_is_reference(label) and not label_is_nonguided(label):
            return label, ds
    raise RuntimeError("No guided parse_raw dataset found among matched generation directories.")


def guidance_spec_from_first_guided(gen_dirs):
    guided_label, parse_raw_ds = first_guided_parse_raw_dataset(gen_dirs)
    callables, targets, guidance_name = callables_from_ds(parse_raw_ds, include_losses=False)
    margins = {key: TARGET_HALF_WIDTH for key in callables}
    return guided_label, {
        "callables": callables,
        "targets": targets,
        "margins": margins,
        "guidance_name": guidance_name,
    }


def count_good(ds, callables, targets, margins) -> int:
    count = 0
    for entry in ds:
        if all(abs(fn(entry) - targets[key]) <= margins[key] for key, fn in callables.items()):
            count += 1
    return count


guided_parse_label, spec = guidance_spec_from_first_guided(GEN_DIRS)
ds_dict = collect_stage_dataset_dict(GEN_DIRS, stage=PVALUE_STAGE, ref_stage=REFERENCE_STAGE_FOR_PLOTS)
print("Datasets at p-value stage:", list(ds_dict))
print("Guidance spec from:", guided_parse_label)
print("Targets:", spec["targets"])
print("Margins:", spec["margins"])

if "Non-guided" not in ds_dict:
    raise RuntimeError("Non-guided dataset was not found. Check that a non-guided generation exists for this system.")

ds_ng = ds_dict["Non-guided"]
rows = []
ng_hits = count_good(ds_ng, spec["callables"], spec["targets"], spec["margins"])
for label, ds_g in ds_dict.items():
    if label_is_reference(label) or label_is_nonguided(label):
        continue
    stats = get_two_proportion_z_test(
        ds_ng,
        spec["callables"],
        spec["targets"],
        spec["margins"],
        ds_guided=ds_g,
        alternative="greater",
    )
    g_hits = count_good(ds_g, spec["callables"], spec["targets"], spec["margins"])
    rows.append({
        "guided_label": label,
        "nonguided_hits": ng_hits,
        "nonguided_total": len(ds_ng),
        "nonguided_fraction": ng_hits / len(ds_ng) if len(ds_ng) else math.nan,
        "guided_hits": g_hits,
        "guided_total": len(ds_g),
        "guided_fraction": g_hits / len(ds_g) if len(ds_g) else math.nan,
        "z_score": stats["z_score"],
        "p_value": stats["p_value"],
    })

pvalue_df = pd.DataFrame(rows)
pvalue_csv = OUTPUT_DIR / f"{SYSTEM}_{GUIDANCE_KIND}_{PVALUE_STAGE}_pvalues.csv"
pvalue_df.to_csv(pvalue_csv, index=False)
display(pvalue_df)
print("Saved:", pvalue_csv)

## 6. Pareto fronts: guidance loss vs. convex-hull energy

This uses the `plot_pareto` helper from `vsbtools.materials_dataset.analysis.pareto_fronts`. It expects the summary-building step above to have created `pf_*.csv` files inside `add_ref_deduplicated` stage directories. The figures are displayed inline and saved as PDFs under the analysis output directory.

In [ ]:
from IPython.display import display

from vsbtools.materials_dataset.analysis.pareto_fronts import plot_pareto

pareto_outputs = []
for pp_stage in PARETO_STAGES:
    for stage_dir in SYSTEM_DIR.rglob(f"{pp_stage}_*"):
        for pf1 in sorted(stage_dir.glob("*pf_1.csv")):
            prefix = pf1.name.replace("pf_1.csv", "")
            loss_name = prefix.strip("_")
            col1 = f"loss_{loss_name}" if loss_name else "loss"
            try:
                ax = plot_pareto(
                    stage_dir,
                    col1=col1,
                    col2="e_hull/at",
                    trim_col1=TRIM_LOSS,
                    trim_col2=TRIM_EHULL,
                    n_fronts=N_PARETO_FRONTS,
                    prefix=prefix,
                    article_axes=True,
                    show_title=False,
                )
            except Exception as exc:
                warnings.warn(f"Skipping {stage_dir} / {pf1.name}: {exc}")
                continue

            out = OUTPUT_DIR / f"{stage_dir.parent.name}_{stage_dir.name}_{prefix}{N_PARETO_FRONTS}_pareto_fronts.pdf"
            ax.figure.savefig(out, bbox_inches="tight", pad_inches=0.1)
            display(ax.figure)
            plt.close(ax.figure)
            pareto_outputs.append(out)

print("Pareto figures:")
for out in pareto_outputs:
    print(" -", out)

## 7. Reproducibility manifest

The manifest records the package paths, bridge configuration, input archives, and analysis parameters used in this run.

In [ ]:
manifest = {
    "python": sys.executable,
    "vsbtools_package": str(VSBTOOLS_PACKAGE),
    "configured_matter_scout_path": str(configured_matter_scout_path) if configured_matter_scout_path else None,
    "configured_grace_python": str(configured_grace_python) if configured_grace_python else None,
    "mattergen_python_path": os.environ.get("MATTERGEN_PYTHON_PATH"),
    "grace_python": os.environ.get("GRACE_PYTHON"),
    "examples_root": str(EXAMPLES_ROOT),
    "scenario_file": str(SCENARIO_FILE),
    "postprocess_root": str(POSTPROCESS_ROOT),
    "processed_root": str(PROCESSED_ROOT),
    "input_dirs": [str(Path(p).expanduser()) for p in INPUT_DIRS],
    "system": SYSTEM,
    "guidance_kind": GUIDANCE_KIND,
    "bond": BOND,
    "target": TARGET,
    "target_half_width": TARGET_HALF_WIDTH,
    "coord_alpha": COORD_ALPHA,
    "hist_stage": HIST_STAGE,
    "pvalue_stage": PVALUE_STAGE,
    "pareto_stages": PARETO_STAGES,
    "output_dir": str(OUTPUT_DIR),
}
manifest_path = OUTPUT_DIR / f"{SYSTEM}_reproducibility_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved:", manifest_path)